## Prototipo: detección de cabecera + extracción de conceptos vía LLM

Objetivo: probar sobre UNA tabla real (Centro Formación, la más
difícil — cabecera en fila 1, con un año suelto en la fila 0) si el
modelo es capaz de:
1. Identificar cuál fila es la cabecera real.
2. Emparejar cada valor de una fila de datos con su concepto.

Si funciona bien aquí, generalizamos al resto de tablas. Si falla,
lo sabemos ahora, con una sola tabla, no después de construir todo
el agente encima.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
import os

modelo = ChatGroq(
    model=os.environ.get("GROQ_MODEL", "llama-3.3-70b-versatile"),
    api_key=os.environ["GROQ_API_KEY"],
    temperature=0,  # nada de creatividad aquí: es extracción, no redacción
)

PROMPT_ANALISTA = """Recibes una tabla en bruto extraída de un documento
municipal, como matriz de filas. Puede tener una fila de título suelta
antes de la cabecera real.

Tu tarea:
1. Identifica qué fila es la cabecera de columnas real.
2. Para la PRIMERA fila de datos (la siguiente a la cabecera), devuelve
   cada par columna->valor que no esté vacío.

Responde SOLO en este formato, una línea por par, sin explicación:
concepto: valor

Tabla:
{tabla}"""

plantilla = ChatPromptTemplate.from_messages([("human", PROMPT_ANALISTA)])

# Tabla más difícil: Centro Formación, año suelto en fila 0
from src_agents.rag.extractor_generico import extraer_xlsx
bloques = extraer_xlsx(DATA_DIR / "financiero_2025_indicadores_control_financiero.xlsx")
tabla_formacion = next(b for b in bloques if b.etiqueta == "Centro Formación")

tabla_texto = "\n".join(str(fila) for fila in tabla_formacion.contenido[:3])
respuesta = modelo.invoke(plantilla.invoke({"tabla": tabla_texto}))
print(respuesta.content)

In [ ]:
print(tabla_formacion.contenido[2])  # la fila de "Enero" tal cual, sin pasar por el LLM

## Validación: detección de cabecera en el caso más difícil

Sobre la tabla más complicada disponible (Centro Formación: fila 0 con
un año suelto, cabecera real en la fila 1), el modelo acertó en las
dos tareas encomendadas:

1. Identificó correctamente que la cabecera no está en la fila 0
   (el "2023"), sino en la fila 1.
2. Emparejó cada columna con su valor de forma fiel — incluyendo
   reportar honestamente que la fila de Enero está vacía en casi
   todas las columnas, en vez de inventar cifras para rellenar.

In [ ]:
# Busca una fila de Centro Formación con datos reales, no la de Enero
for fila in tabla_formacion.contenido[1:]:
    if any(v is not None for v in fila[1:]):
        print(fila)
        break

### Ojo — esto no es una fila de datos, es la cabecera otra vez

El bucle empezaba en `contenido[1:]`, pero `contenido[1]` es la fila de
cabecera (`MES`, `Nº CURSOS...`), no una fila de Enero/Febrero con
valores. Como los nombres de columna también son texto ("no None"),
la condición `any(v is not None for v in fila[1:])` se cumplió en la
propia cabecera y el bucle paró ahí, sin llegar a ninguna fila real.

In [ ]:
for fila in tabla_formacion.contenido[2:]:
    if any(v is not None for v in fila[1:]):
        print(fila)
        break

### Buen caso de prueba — y con una trampa a propósito

Fíjate: `16` cae en el índice 5, que es `Nº CONTRATADOS PIL`. Pero hay
algo más interesante en esta cabecera — el nombre `Nº ALUMNOS` aparece
**dos veces** (índice 2 y 4), una vez para el curso presencial y otra
para el online. Si el modelo no distingue por posición y solo empareja
por nombre de columna, "Nº ALUMNOS" es ambiguo — no sabría a cuál de
los dos se refiere. Este caso nos sirve para comprobar precisamente eso.

In [ ]:
fila_diciembre = "\n".join(str(f) for f in [tabla_formacion.contenido[1], tabla_formacion.contenido[8]])
# fila 1 = cabecera, fila 8 = la de Diciembre (ajusta el índice si no coincide en tu notebook)

respuesta = modelo.invoke(plantilla.invoke({"tabla": fila_diciembre}))
print(respuesta.content)

### Qué buscar en la respuesta

1. Que el valor `16` quede asociado a "Nº CONTRATADOS PIL", no a
   cualquiera de los "Nº ALUMNOS".
2. Que no aparezca un "16" fantasma en ninguna otra columna.
3. Si el modelo devuelve las dos columnas "Nº ALUMNOS" con el mismo
   nombre sin diferenciar cuál es cuál, tenemos un problema real de
   ambigüedad que hay que resolver en el prompt (dándole también el
   índice de columna, no solo el nombre) antes de construir el agente
   completo.

### Algo no cuadra — y hay que pararse en esto, no seguir

El prompt le mandó al modelo la fila 8, asumiendo que era la de
Diciembre (la que tenía el `16` en `Nº CONTRATADOS PIL`). Pero la
respuesta dice `MES: Julio`, con todo vacío — es decir, `contenido[8]`
NO es la fila de Diciembre en tu notebook. El índice que asumí
(fila 1 = cabecera, fila 8 = Diciembre) no coincide con el orden real
de las filas en esta tabla. No es un fallo del modelo: le dimos la
fila equivocada, y él reportó fielmente lo que había en ella (vacía,
como ya viste que pasa con varios meses).

### Vamos a localizar la fila correcta por contenido, no por índice adivinado

### Por qué así, no por índice

Buscar por `f[0] == "Diciembre"` no depende de contar filas a mano y
adivinar dónde cae cada mes — si la tabla tiene alguna fila más o
menos de las que yo conté, este método sigue encontrando la correcta.
Es el mismo principio que ya aplicasteis con el forward-fill: apoyarse
en el dato real, no en una posición asumida.

In [ ]:
fila_diciembre_real = next(f for f in tabla_formacion.contenido if f[0] == "Diciembre")
print(fila_diciembre_real)

### Ahora sí, el test que buscábamos

Mandamos al modelo la cabecera real (`tabla_formacion.contenido[1]`)
junto con la fila de Diciembre correcta. Lo que nos interesa: ¿el `16`
queda bien atado a "Nº CONTRATADOS PIL", y el modelo distingue entre
las dos columnas "Nº ALUMNOS" (presencial vs online) sin confundirlas,
o le sale ambiguo por tener el mismo nombre dos veces?

In [ ]:
cabecera = tabla_formacion.contenido[1]
fila_diciembre = tabla_formacion.contenido.index(fila_diciembre_real)  # índice real, no adivinado
tabla_texto = "\n".join(str(f) for f in [cabecera, fila_diciembre_real])

respuesta = modelo.invoke(plantilla.invoke({"tabla": tabla_texto}))
print(respuesta.content)

### El modelo resolvió bien la ambigüedad

`Nº CONTRATADOS PIL: 16` — correcto, esa es la columna donde vive el
valor real. Y aunque aparecen las dos "Nº ALUMNOS" en la respuesta,
NO se confunden entre sí ni se les asigna el 16 por error — ambas
quedan vacías, que es justo lo que corresponde, porque en la fila de
Diciembre solo hay dato en `Nº CONTRATADOS PIL`.

No hemos visto todavía un caso donde AMBAS columnas "Nº ALUMNOS"
tengan valor a la vez en la misma fila — ese sería el test definitivo
de si el modelo las distingue de verdad por posición o si por
casualidad ha acertado porque solo una tenía dato. Vale la pena
tenerlo en mente si en el futuro aparece una fila así y el resultado
no cuadra.

### Ajuste de prompt: de una fila a la tabla completa

Antes: "para la PRIMERA fila de datos, devuelve cada par columna->valor"
Ahora: "para CADA fila de datos, devuelve sus pares columna->valor,
separando cada fila con una línea en blanco y precedida por el
identificador de esa fila (ej. el mes, si la primera columna lo es)"

Esto mantiene el mismo formato de parseo simple (`concepto: valor`)
que ya viste que funciona, solo repetido por bloques.

### Prototipo: tabla completa (Empleo), no solo una fila

Objetivo: comprobar que, con la cabecera + TODAS las filas de una
tabla real, el modelo mantiene cada fila separada y no mezcla datos
entre meses (por ejemplo, que un valor de Marzo no acabe atribuido
a Febrero). Empleo es buena tabla de prueba: cabecera a dos niveles
ya resuelta por el fix, y varias filas con datos reales distintos
entre sí.

In [ ]:
PROMPT_ANALISTA_TABLA = """Recibes una tabla en bruto extraída de un documento
municipal, como matriz de filas (la primera es la cabecera de columnas).

Tu tarea: para CADA fila de datos (todas menos la cabecera), devuelve
los pares columna->valor que tengan un valor real (puedes omitir o
dejar en blanco los que no lo tengan, cualquiera de las dos formas
está bien).

Antes de cada fila, escribe una línea con el identificador de esa fila
(el valor de la primera columna, ej. el mes). Separa cada fila con una
línea en blanco. No mezcles datos entre filas distintas.

Responde en este formato, sin explicación adicional:
--- <identificador de fila> ---
concepto: valor
concepto: valor

Tabla:
{tabla}"""

plantilla_tabla = ChatPromptTemplate.from_messages([("human", PROMPT_ANALISTA_TABLA)])

bloques_empleo = extraer_xlsx(DATA_DIR / "financiero_2025_indicadores_control_financiero.xlsx")
tabla_empleo = next(b for b in bloques_empleo if b.etiqueta == "Empleo")

tabla_texto_completa = "\n".join(str(fila) for fila in tabla_empleo.contenido)
print(f"Filas totales en la tabla (incluida cabecera): {len(tabla_empleo.contenido)}")

respuesta_tabla = modelo.invoke(plantilla_tabla.invoke({"tabla": tabla_texto_completa}))
print(respuesta_tabla.content)

### Qué revisar en la salida

1. ¿Aparecen tantos bloques `--- <mes> ---` como filas de datos tenía
   la tabla (cabecera aparte)? Si faltan meses, el modelo se "comió"
   filas al procesar el bloque completo — señal de que hay que acotar
   cuántas filas mandar por llamada.
2. Coge un valor cualquiera de una fila (por ejemplo, "TOTAL CONTRATOS"
   de un mes concreto) y compáralo a mano contra `tabla_empleo.contenido`
   para ese mes — confirma que no se ha desplazado a otro mes.
3. ¿El formato de salida es lo bastante regular como para parsearlo con
   una expresión simple (separar por `--- ---`, luego por líneas
   `concepto: valor`), o el modelo improvisa el formato de forma que
   rompería un parser sencillo?

### Resultado limpio — el enfoque de "una llamada por tabla" aguanta

Repaso de los 3 puntos que había que revisar:

1. **¿Faltan filas?** No — 16 bloques de datos (Enero a Diciembre + Q1,
   Q2, Q3 + TOTALES), que es exactamente lo que tenía la tabla. Ninguna
   fila se perdió al procesar el bloque completo.
2. **¿Se desplazó algún valor entre meses?** Comprobado con Enero contra
   el dato en bruto que ya habíamos visto en el notebook 01
   (`['Enero', 360, 37, 46, 266, 3, 0, 110, 333, 10, 4, None, 60, 74]`):
   coincide cifra a cifra, incluida la columna vacía
   ("Información telefónica") que el modelo omitió en vez de inventar
   un valor.
3. **¿Formato parseable?** Sí — bloques `--- <identificador> ---`
   seguidos de líneas `concepto: valor`, consistente en las 16 filas.
   Un `split` por `---` y luego por `\n` lo procesa sin ambigüedad.

### Algo a anotar para el diseño del Analista — no es un fallo, es un matiz real

Fíjate en `Q1`, `Q2`, `Q3` y `TOTALES`: son filas reales de la tabla,
así que el modelo las trató igual que un mes (`MES: Q1`) — hizo bien
en no inventarse una categoría distinta, porque nosotros no se lo
pedimos. Pero para el Redactor, "Q1" no es un mes, es un agregado
trimestral, y si acaba en el informe como si fuera uno más, sería
confuso ("en el mes de Q1 hubo 16 contratos").

Esto no lo arregla el prompt del Analista — el Analista está haciendo
bien su trabajo (reportar fielmente lo que hay). Es una decisión para
el Redactor o para el propio `ConceptoValor`: o bien el Analista marca
estas filas como agregados en vez de meses (ajuste de prompt), o el
Redactor recibe la instrucción de tratar Q1/Q2/Q3/TOTALES como
resúmenes de periodo, no como meses individuales.

## Conclusión del notebook 04 — prototipo del Agente Analista

**Objetivo del notebook:** comprobar, antes de escribir el agente
completo, si un LLM (Groq, llama-3.3-70b-versatile) puede interpretar
tablas reales del Ayuntamiento — detectar la cabecera correcta y
emparejar cada valor con su concepto — sin la mala atribución que
sufrió el pipeline RAG anterior con tablas densas.

### Lo que se probó, en orden

1. **Una fila vacía** (Enero, hoja Centro Formación) — el modelo
   reportó fielmente que no había datos, sin inventar cifras.
2. **Una fila con dato único entre columnas de nombre repetido**
   (Diciembre, dos columnas "Nº ALUMNOS") — el modelo atribuyó el
   valor a la columna correcta sin confundir las dos homónimas.
3. **Una tabla completa, 16 filas de datos** (hoja Empleo) — ninguna
   fila se perdió, ningún valor se desplazó entre meses, formato de
   salida consistente y fácil de parsear.

### Decisión de arquitectura resultante

- **Una llamada al modelo por tabla completa**, no por fila — más
  barato en número de llamadas, más contexto disponible para el
  modelo (la cabecera completa siempre presente), y más defendible
  como decisión de diseño que "una llamada por fila".
- **El Analista no distingue tipos de fila** (mes vs. trimestre vs.
  total) — reporta cada fila tal como aparece en la tabla, sin
  interpretación adicional. Esa distinción semántica se resuelve
  aguas abajo, en el Redactor, que recibe instrucción explícita de
  tratar Q1/Q2/Q3/TOTALES como resúmenes de periodo y no como meses.
- **Sigue habiendo supervisión humana del informe final** antes de
  entregarlo — el diseño reduce el riesgo de mala atribución, no lo
  elimina, y esa revisión humana sigue siendo la última red de
  seguridad, igual que en la fase RAG.

### Por qué esto resuelve el problema original

En la fase RAG, un único paso tenía que interpretar la estructura de
la tabla Y redactar al mismo tiempo, y fallaba en tablas de 13
columnas. Aquí, el Analista solo interpreta estructura — nunca
redacta — y el prototipo confirma que, aislado así, lo hace bien
incluso en el caso más difícil que tenéis (cabecera fusionada a dos
niveles, columnas con nombre repetido).

### Diseño de analyst.py, antes del código

El Analista recibe `state["documents"]` (todos los BloqueContenido que
produce el extractor) y devuelve `state["analysis"]`, un `Analisis`
con la lista de `ConceptoValor` que ya definió tu compañera.

Una decisión que dejo explícita: **este primer analyst.py solo procesa
bloques de tipo "tabla"**, no los de tipo "texto" (el narrativo del
docx, con cifras incrustadas como "1.396 personas"). No lo hemos
probado todavía — sería una generalización sin validar, y ya sabéis
cómo de mal sale eso. Lo dejo anotado como pendiente, no como "ya
resuelto".

In [ ]:
# Celda — definición completa del Analista
import re
from src_agents.models.state import Analisis, ConceptoValor


def _parsear_respuesta(texto: str, etiqueta_tabla: str, fuente: str) -> list[ConceptoValor]:
    """Convierte la respuesta del modelo (bloques --- id --- + concepto: valor)
    en una lista de ConceptoValor. Ignora líneas sin valor y las que no
    siguen el formato 'concepto: valor' (robusto frente a que el modelo
    no obedezca al pie de la letra el formato pedido, como ya viste)."""
    conceptos = []
    identificador_actual = None
    for linea in texto.splitlines():
        linea = linea.strip()
        if not linea:
            continue
        m_id = re.match(r"^---\s*(.+?)\s*---$", linea)
        if m_id:
            identificador_actual = m_id.group(1)
            continue
        if ":" not in linea:
            continue
        concepto, _, valor = linea.partition(":")
        concepto, valor = concepto.strip(), valor.strip()
        if not valor:
            continue  # celda vacía, no un dato real
        nombre = f"{concepto} ({identificador_actual})" if identificador_actual else concepto
        conceptos.append(ConceptoValor(concepto=nombre, valor=valor, fuente=f"{fuente} - {etiqueta_tabla}"))
    return conceptos


def agente_analista_prototipo(bloques_tabla) -> Analisis:
    todos_los_conceptos = []
    notas = []
    for bloque in bloques_tabla:
        tabla_texto = "\n".join(str(fila) for fila in bloque.contenido)
        respuesta = modelo.invoke(plantilla_tabla.invoke({"tabla": tabla_texto}))
        conceptos = _parsear_respuesta(respuesta.content, bloque.etiqueta, bloque.fuente)
        if not conceptos:
            notas.append(f"Sin datos interpretables en '{bloque.etiqueta}' ({bloque.fuente})")
        todos_los_conceptos.extend(conceptos)
    return Analisis(datos=todos_los_conceptos, notas="; ".join(notas))

In [ ]:
# Celda — prueba contra un documento COMPLETO, no una sola hoja
bloques_indicadores = extraer_xlsx(DATA_DIR / "financiero_2025_indicadores_control_financiero.xlsx")
print(f"Total bloques (hojas) en el archivo: {len(bloques_indicadores)}")

analisis_resultado = agente_analista_prototipo(bloques_indicadores)
print(f"Total ConceptoValor extraídos: {len(analisis_resultado.datos)}")
print(f"Notas: {analisis_resultado.notas or '(sin notas)'}")
print()
print("Primeros 5 conceptos:")
for c in analisis_resultado.datos[:5]:
    print(f"  {c.concepto} = {c.valor}  [{c.fuente}]")

### 7 hojas procesadas, ninguna en notas (todas dieron datos), y sin fallos silenciosos. 820 conceptos es un volumen razonable para 7 hojas con hasta 19 filas cada una. Los 5 primeros coinciden exactamente con lo que ya validamos a mano para Enero.

In [ ]:
conceptos_incidencias = [c for c in analisis_resultado.datos if "Incidencias" in c.fuente]
print(f"Conceptos de Incidencias informáticas: {len(conceptos_incidencias)}")
for c in conceptos_incidencias[:5]:
    print(f"  {c.concepto} = {c.valor}")

In [ ]:
bloque_incidencias = next(b for b in bloques_indicadores if b.etiqueta == "Incidencias informáticas")
print(f"Filas reales con datos (tras filtrar vacías): {len(bloque_incidencias.contenido)}")

### Confirmado: truncamiento real, no falta de datos

57 filas reales (cabecera aparte, ~56 incidentes) frente a solo 24
conceptos extraídos (~6 incidentes representados). La hipótesis se
confirma: el modelo trunca su respuesta cuando la tabla es grande.
Con 18 filas (Empleo) no pasó — el límite está en algún punto entre
18 y 57 filas por llamada.

Sobre la privacidad: quedo con que el Ayuntamiento ya está al tanto
de que se usa un modelo en la nube y ha confirmado que no hay datos
sensibles en estos documentos — con eso, seguimos sin bloquear nada
por ese lado.

### Siguiente paso: trocear las tablas grandes antes de mandarlas

"Una llamada por tabla completa" sigue siendo la decisión correcta
para tablas normales (Empleo, con 18 filas, funcionó perfecto) — el
ajuste es solo para tablas que superen un umbral de filas: partirlas
en bloques de N filas (manteniendo la cabecera repetida en cada
bloque, para que el modelo no pierda contexto de columnas) y unir
los resultados después.

### Troceo de tablas grandes por bloques de filas

Cada bloque repite la cabecera (fila 0) para que el modelo no pierda
el contexto de qué columna es cada una — sin la cabecera, un bloque
intermedio de una tabla de 57 filas sería solo números sueltos, sin
forma de saber a qué concepto pertenece cada uno.

In [ ]:
# Celda — función de troceo + integración en el Analista
UMBRAL_FILAS = 20

def _trocear_tabla(contenido: list[list], tamano_bloque: int = UMBRAL_FILAS) -> list[list[list]]:
    """Divide una tabla grande en bloques de tamano_bloque filas de datos,
    repitiendo la cabecera (fila 0) en cada bloque para que el modelo
    conserve el contexto de columnas."""
    if len(contenido) <= tamano_bloque:
        return [contenido]
    cabecera = contenido[0]
    filas_datos = contenido[1:]
    bloques = []
    for i in range(0, len(filas_datos), tamano_bloque - 1):  # -1: la cabecera ocupa una "fila" del bloque
        trozo = filas_datos[i:i + tamano_bloque - 1]
        bloques.append([cabecera] + trozo)
    return bloques


def agente_analista_prototipo(bloques_tabla) -> Analisis:
    todos_los_conceptos = []
    notas = []
    for bloque in bloques_tabla:
        sub_tablas = _trocear_tabla(bloque.contenido)
        for sub_tabla in sub_tablas:
            tabla_texto = "\n".join(str(fila) for fila in sub_tabla)
            respuesta = modelo.invoke(plantilla_tabla.invoke({"tabla": tabla_texto}))
            conceptos = _parsear_respuesta(respuesta.content, bloque.etiqueta, bloque.fuente)
            if not conceptos:
                notas.append(f"Sin datos interpretables en un bloque de '{bloque.etiqueta}' ({bloque.fuente})")
            todos_los_conceptos.extend(conceptos)
    return Analisis(datos=todos_los_conceptos, notas="; ".join(notas))

In [ ]:
# Celda — vuelve a probar SOLO Incidencias informáticas, ahora troceada
analisis_incidencias = agente_analista_prototipo([bloque_incidencias])
print(f"Conceptos extraídos ahora: {len(analisis_incidencias.datos)}")
print(f"Notas: {analisis_incidencias.notas or '(sin notas)'}")

### Prototipo cerrado y validado en los tres frentes que importaban: tabla pequeña (Empleo), tabla mediana con ambigüedad de nombres (Centro Formación), tabla grande (Incidencias informáticas).

In [ ]:
from src_agents.rag.extractor_generico import extraer_carpeta
from src_agents.agents.analyst import agente_analista

todos_los_bloques = extraer_carpeta(DATA_DIR)
resultado = agente_analista({"documents": todos_los_bloques})
print(f"Total ConceptoValor: {len(resultado['analysis'].datos)}")
print(f"Notas: {resultado['analysis'].notas or '(sin notas)'}")

### No hay más tokens ;(